# File and Project Overview

This notebook belongs to the `phase1` implementation of the Fake Review Detection System.

Project goal:
- Predict whether a review is `fake` or `genuine`
- Return a `fake probability` score for each review

About this file (`phase1_v3_colab.ipynb`):
- This is the **V3 blended model** notebook
- It trains two models (text model + metadata model) and blends probabilities with a learned weight (`alpha`)
- It includes full flow: install deps, load data, clean/split, feature build, train/evaluate, save artifacts, and sample prediction

Dataset used in this phase:
- `dataset/amazon_labeled_fake_reviews/final_labeled_fake_reviews.csv`

How to run:
- Execute cells from top to bottom
- Keep output artifact paths as-is for Colab, or change if needed for local runs

# Phase 1 v3 - Fake Review Detection (Colab)

This notebook is a complete v3 (blended text + metadata) training pipeline in one place.

Run cells top-to-bottom.

In [ ]:
# Install required packages (safe to re-run).
!pip -q install pandas numpy scikit-learn nltk scipy joblib

In [ ]:
# Imports and notebook-level config.
import os
import re
import json
import hashlib
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, brier_score_loss
from scipy.sparse import csr_matrix, hstack

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Imports loaded.')

In [ ]:
# Load dataset (local path first, then Colab upload fallback).
LOCAL_DATA_PATH = 'dataset/amazon_labeled_fake_reviews/final_labeled_fake_reviews.csv'
COLAB_DATA_PATH = '/content/final_labeled_fake_reviews.csv'

if os.path.exists(LOCAL_DATA_PATH):
    DATA_PATH = LOCAL_DATA_PATH
elif os.path.exists(COLAB_DATA_PATH):
    DATA_PATH = COLAB_DATA_PATH
else:
    try:
        from google.colab import files
        print('Upload final_labeled_fake_reviews.csv ...')
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise ValueError('No file uploaded')
        DATA_PATH = '/content/' + list(uploaded.keys())[0]
    except Exception as ex:
        raise RuntimeError('Set DATA_PATH manually if local/Colab auto-detect fails.') from ex

df_raw = pd.read_csv(DATA_PATH)
print('Using DATA_PATH:', DATA_PATH)
print('Rows:', len(df_raw))
print('Columns:', list(df_raw.columns))

In [ ]:
# Cleaning, dedupe, and leakage-safe split helpers.
def normalize_label(value):
    # Normalize labels to integer 0/1.
    if pd.isna(value):
        return np.nan
    try:
        return int(value)
    except Exception:
        value_str = str(value).strip().lower()
        if value_str in {'fake', '1', 'true', 'yes', 'y'}:
            return 1
        if value_str in {'genuine', 'real', '0', 'false', 'no', 'n'}:
            return 0
        return np.nan

def basic_text_clean(text):
    # Normalize text for dedupe/split hashing.
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def make_text_hash(text_value):
    # Build deterministic group hash.
    return hashlib.md5(text_value.encode('utf-8')).hexdigest()

df = pd.DataFrame()
df['text'] = df_raw['text'].astype(str)
df['label'] = df_raw['label'].apply(normalize_label)
df['rating'] = pd.to_numeric(df_raw.get('rating'), errors='coerce').fillna(0.0)
df['helpful_vote'] = pd.to_numeric(df_raw.get('helpful_vote'), errors='coerce').fillna(0.0)
df['verified_purchase'] = df_raw.get('verified_purchase', pd.Series(['FALSE'] * len(df_raw))).astype(str).str.upper().map({'TRUE':1,'FALSE':0}).fillna(0).astype(int)
df['text_clean_for_split'] = df['text'].apply(basic_text_clean)

# Remove invalid rows.
df = df[df['label'].isin([0,1])]
df = df[df['text_clean_for_split'].str.len() > 0]

# Dedupe before split to reduce leakage.
df = df.drop_duplicates(subset=['text_clean_for_split']).reset_index(drop=True)
df['text_hash'] = df['text_clean_for_split'].apply(make_text_hash)

group_df = df.groupby('text_hash', as_index=False)['label'].first()
keys = group_df['text_hash'].values
y_group = group_df['label'].values

# Split hash groups: train/calibration/validation/test.
train_calib_val_keys, test_keys, y_train_calib_val, _ = train_test_split(
    keys, y_group, test_size=0.10, stratify=y_group, random_state=RANDOM_SEED
)

train_val_keys, calib_keys, y_train_val, _ = train_test_split(
    train_calib_val_keys,
    y_train_calib_val,
    test_size=(0.10 / 0.90),
    stratify=y_train_calib_val,
    random_state=RANDOM_SEED
)

train_keys, val_keys, _, _ = train_test_split(
    train_val_keys,
    y_train_val,
    test_size=(0.10 / 0.80),
    stratify=y_train_val,
    random_state=RANDOM_SEED
)

split_map = {
    'train': set(train_keys.tolist()),
    'calibration': set(calib_keys.tolist()),
    'validation': set(val_keys.tolist()),
    'test': set(test_keys.tolist())
}

df['split'] = 'unassigned'
for split_name, key_set in split_map.items():
    df.loc[df['text_hash'].isin(key_set), 'split'] = split_name

assert (df['split'] != 'unassigned').all()
print(df['split'].value_counts())
print('Label mean by split:\n', df.groupby('split')['label'].mean())

In [ ]:
# Build v3 feature sets (text model features + metadata model features).
def clean_text_for_model(text):
    # Normalize text for TF-IDF.
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r"[^a-z0-9\s!?']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def punctuation_ratio(text):
    # Compute punctuation density.
    if not text:
        return 0.0
    punct_count = sum(1 for c in text if c in '!?.,:;\"\'()-[]{}')
    return punct_count / max(len(text), 1)

def uppercase_ratio(text):
    # Compute uppercase ratio.
    if not text:
        return 0.0
    upper = sum(1 for c in text if c.isupper())
    alpha = sum(1 for c in text if c.isalpha())
    return upper / max(alpha, 1)

def build_text_numeric_features(frame):
    # Build text-style numeric features for text model.
    out = pd.DataFrame(index=frame.index)
    raw_text = frame['text'].fillna('').astype(str)
    clean = frame['text_model'].fillna('').astype(str)

    out['char_count'] = raw_text.str.len()
    out['word_count'] = clean.str.split().str.len().fillna(0)
    out['avg_word_len'] = out['char_count'] / np.maximum(out['word_count'], 1)
    out['exclamation_count'] = raw_text.str.count('!')
    out['question_count'] = raw_text.str.count(r'\?')
    out['punctuation_ratio'] = raw_text.apply(punctuation_ratio)
    out['uppercase_ratio'] = raw_text.apply(uppercase_ratio)
    return out.astype(float)

def build_behavioral_matrix(frame):
    # Build behavioral-only matrix for metadata model.
    out = pd.DataFrame(index=frame.index)
    out['rating'] = pd.to_numeric(frame.get('rating'), errors='coerce').fillna(0.0)
    out['helpful_vote'] = pd.to_numeric(frame.get('helpful_vote'), errors='coerce').fillna(0.0)
    out['verified_purchase'] = pd.to_numeric(frame.get('verified_purchase'), errors='coerce').fillna(0.0)
    return out.values

split_frames = {k: v.copy() for k, v in df.groupby('split')}
for k in split_frames:
    split_frames[k]['text_model'] = split_frames[k]['text'].apply(clean_text_for_model)

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=0.98,
    max_features=50000,
    sublinear_tf=True
)
vectorizer.fit(split_frames['train']['text_model'])

def make_text_matrix(frame):
    # Build sparse text-model matrix (tfidf + text numeric).
    x_text = vectorizer.transform(frame['text_model'])
    x_num = build_text_numeric_features(frame)
    return x_text, x_num

x_train_text, train_num = make_text_matrix(split_frames['train'])
x_val_text, val_num = make_text_matrix(split_frames['validation'])
x_test_text, test_num = make_text_matrix(split_frames['test'])

num_mean = train_num.mean(axis=0)
num_std = train_num.std(axis=0).replace(0, 1.0)

def scale_num(num):
    # Scale text numeric features with train statistics.
    return (num - num_mean) / num_std

x_train = hstack([x_train_text, csr_matrix(scale_num(train_num).values)], format='csr')
x_val = hstack([x_val_text, csr_matrix(scale_num(val_num).values)], format='csr')
x_test = hstack([x_test_text, csr_matrix(scale_num(test_num).values)], format='csr')

x_meta_train = build_behavioral_matrix(split_frames['train'])
x_meta_val = build_behavioral_matrix(split_frames['validation'])
x_meta_test = build_behavioral_matrix(split_frames['test'])

y_train = split_frames['train']['label'].astype(int).values
y_val = split_frames['validation']['label'].astype(int).values
y_test = split_frames['test']['label'].astype(int).values

print('Text train matrix shape:', x_train.shape)
print('Metadata train matrix shape:', x_meta_train.shape)

In [ ]:
# Train v3 blend and evaluate.
def expected_calibration_error(y_true, y_prob, bins=10):
    # Compute ECE from probability bins.
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def eval_metrics(y_true, y_prob, threshold):
    # Compute summary metrics for blended probabilities.
    pred = (y_prob >= threshold).astype(int)
    return {
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'brier': float(brier_score_loss(y_true, y_prob)),
        'ece_10': float(expected_calibration_error(y_true, y_prob, bins=10)),
        'confusion_matrix': confusion_matrix(y_true, pred).tolist()
    }

def grid_search_alpha_threshold(y_true, text_prob, meta_prob):
    # Search blend alpha and threshold by validation F1.
    best = {'alpha': 0.85, 'threshold': 0.5, 'f1': -1.0}
    for alpha in np.linspace(0.5, 0.95, 10):
        blend = alpha * text_prob + (1.0 - alpha) * meta_prob
        for threshold in np.linspace(0.2, 0.8, 25):
            pred = (blend >= threshold).astype(int)
            curr_f1 = f1_score(y_true, pred, zero_division=0)
            if curr_f1 > best['f1']:
                best = {
                    'alpha': float(alpha),
                    'threshold': float(threshold),
                    'f1': float(curr_f1),
                }
    return best

# Text-focused logistic model.
text_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)
text_model.fit(x_train, y_train)
text_prob_val = text_model.predict_proba(x_val)[:, 1]
text_prob_test = text_model.predict_proba(x_test)[:, 1]

# Metadata-only logistic model.
meta_scaler = StandardScaler()
x_meta_train_scaled = meta_scaler.fit_transform(x_meta_train)
meta_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED)
meta_model.fit(x_meta_train_scaled, y_train)
meta_prob_val = meta_model.predict_proba(meta_scaler.transform(x_meta_val))[:, 1]
meta_prob_test = meta_model.predict_proba(meta_scaler.transform(x_meta_test))[:, 1]

# Learn blend parameters on validation split.
best = grid_search_alpha_threshold(y_val, text_prob_val, meta_prob_val)
alpha = best['alpha']
threshold = best['threshold']

val_blend = alpha * text_prob_val + (1.0 - alpha) * meta_prob_val
test_blend = alpha * text_prob_test + (1.0 - alpha) * meta_prob_test

val_metrics = eval_metrics(y_val, val_blend, threshold)
test_metrics = eval_metrics(y_test, test_blend, threshold)

print('Selected alpha (text weight):', alpha)
print('Selected threshold:', threshold)
print('Validation F1:', val_metrics['f1'])
print('Test F1:', test_metrics['f1'])
print('Test metrics:', test_metrics)

In [ ]:
# Save v3 artifacts and run sample prediction.
OUT_DIR = '/content/phase1_v3_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(text_model, os.path.join(OUT_DIR, 'text_model.joblib'))
joblib.dump(vectorizer, os.path.join(OUT_DIR, 'text_vectorizer.joblib'))
joblib.dump(meta_model, os.path.join(OUT_DIR, 'meta_model.joblib'))
joblib.dump(meta_scaler, os.path.join(OUT_DIR, 'meta_scaler.joblib'))

feature_metadata = {
    'numeric_columns': list(train_num.columns),
    'numeric_mean': num_mean.to_dict(),
    'numeric_std': num_std.to_dict(),
    'include_behavioral': False
}
with open(os.path.join(OUT_DIR, 'feature_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(feature_metadata, f, indent=2)

model_metadata = {
    'model_name': 'blend_text_and_metadata',
    'model_version': 'phase1-v3-colab',
    'blend_weight_text': float(alpha),
    'blend_weight_metadata': float(1.0 - alpha),
    'threshold': float(threshold),
    'random_seed': int(RANDOM_SEED),
    'include_behavioral': 'blended'
}
with open(os.path.join(OUT_DIR, 'model_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2)

def predict_one(text, rating=0, helpful_vote=0, verified_purchase=0):
    # Predict fake probability using v3 text+metadata blend.
    frame = pd.DataFrame([{
        'text': text,
        'rating': rating,
        'helpful_vote': helpful_vote,
        'verified_purchase': verified_purchase
    }])
    frame['text_model'] = frame['text'].apply(clean_text_for_model)

    x_text_only = vectorizer.transform(frame['text_model'])
    x_text_num = build_text_numeric_features(frame)
    x_text_scaled = scale_num(x_text_num)
    x_text = hstack([x_text_only, csr_matrix(x_text_scaled.values)], format='csr')

    x_meta = build_behavioral_matrix(frame)

    prob_text = float(text_model.predict_proba(x_text)[:, 1][0])
    prob_meta = float(meta_model.predict_proba(meta_scaler.transform(x_meta))[:, 1][0])
    prob_blend = float(alpha * prob_text + (1.0 - alpha) * prob_meta)

    label = 'fake' if prob_blend >= threshold else 'genuine'
    return {
        'label': label,
        'fake_probability': prob_blend,
        'fake_percent': round(prob_blend * 100, 2),
        'threshold': threshold,
        'text_probability': prob_text,
        'metadata_probability': prob_meta,
        'alpha_text': alpha,
        'alpha_metadata': float(1.0 - alpha)
    }

example = predict_one(
    text='I used this for two weeks. Build quality is decent and setup was easy, but battery lasts around 5 hours instead of the 7 listed.',
    rating=3,
    helpful_vote=2,
    verified_purchase=1
)
print('Example prediction:', example)
print('Artifacts saved in:', OUT_DIR)